# Examine output

use this notebook to see how effective the processing is for 2022.

In [ ]:
import requests

vars2021 = requests.get("https://api.census.gov/data/2021/acs/acs5/variables.json").json()["variables"]
vars2022 = requests.get("https://api.census.gov/data/2022/acs/acs5/variables.json").json()["variables"]

In [ ]:
missing = newly_missing_in_2022

still_exist = []
gone = []

for var in missing:
    if var in vars2022:
        still_exist.append(var)
    else:
        gone.append(var)

print("Still exist in 2022:", len(still_exist))
print("Gone in 2022:", len(gone))

So like, the variables still exist per the metadata (variables json provided by the ACS), but they are not present where I expect them (tiger product). First, let's ID all the variables by their census identifiers:

In [ ]:
for var in still_exist:
    print(var, "->", vars2022[var]["label"])

In [ ]:
import re
from collections import defaultdict

In [ ]:
def variable_to_table_group(var: str) -> str | None:
    """
    Convert an ACS variable like B01003_001E to its table/group name B01003.
    """
    m = re.match(r"^([A-Z0-9]+)_\d+[A-Z]$", var)
    if m:
        return m.group(1)
    return None


def group_variables_by_table(vars_list: list[str]) -> dict[str, list[str]]:
    groups = defaultdict(list)
    unparsed = []

    for var in sorted(set(vars_list)):
        group = variable_to_table_group(var)
        if group is None:
            unparsed.append(var)
        else:
            groups[group].append(var)

    if unparsed:
        print("Could not parse these variables:")
        for var in unparsed:
            print("  ", var)

    return dict(sorted(groups.items()))


groups = group_variables_by_table(newly_missing_in_2022)

print(f"Unique table groups: {len(groups)}")
for group, vars_ in groups.items():
    print(f"{group}: {len(vars_)} vars")

## Another way to inspect the tables/groups

In [ ]:
def describe_group(group: str, vars_meta: dict) -> pd.DataFrame:
    rows = []
    for var, meta in vars_meta.items():
        if var.startswith(f"{group}_"):
            rows.append(
                {
                    "variable": var,
                    "label": meta.get("label"),
                    "concept": meta.get("concept"),
                    "predicateType": meta.get("predicateType"),
                    "group": meta.get("group"),
                }
            )
    return pd.DataFrame(rows).sort_values("variable")

In [ ]:
# Example:
describe_group("B01001", vars2022).head(20)

## Reclaim new naming format

Follow Eli's comment on the PR:

`ok, now that i've looked at ont of the 2022 tables in the geodatabase, the reason you're getting no results is the naming convention has changed. Your PR includes an update for the geoid column, but there are other systematic changes. In the new tables, the variables are named (as an example): B02001_E001. We need to have processing that anticipates this format, then converts it to the canonical form (like the json tables, B02001_001E (where E/M is the final character of the variable rather than the leading character)`

In [1]:
import pandas as pd
import geopandas as gpd

import re
from pathlib import Path
import pyarrow.parquet as pq

from IPython.display import display

In [2]:
# Adjust this path if needed
BUILD_ROOT = Path("../build")

DIR_2021 = BUILD_ROOT / "2021_bg"
DIR_2022 = BUILD_ROOT / "2022_bg"
REPORT_DIR = BUILD_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("2021 dir:", DIR_2021.resolve())
print("2022 dir:", DIR_2022.resolve())
print("Report dir:", REPORT_DIR.resolve())

2021 dir: /home/dylan/projects/geosnap/build/2021_bg
2022 dir: /home/dylan/projects/geosnap/build/2022_bg
Report dir: /home/dylan/projects/geosnap/build/reports


## Inspect the new naming on one file

In [3]:
test1 = pd.read_parquet(f'{DIR_2022}/acs_2022_X14_SCHOOL_ENROLLMENT_bg.parquet')

In [4]:
test1.head()

,B14002_E001,B14002_E002,B14002_E003,B14002_E004,B14002_E005,B14002_E006,B14002_E007,B14002_E008,B14002_E009,B14002_E010,...,B14007I_E010,B14007I_E011,B14007I_E012,B14007I_E013,B14007I_E014,B14007I_E015,B14007I_E016,B14007I_E017,B14007I_E018,B14007I_E019
GEOIDFQ,,,,,,,,,,,,,,,,,,,,,
1500000US010179548002,1375.0,529.0,45.0,24.0,24.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,21.0
1500000US010179548004,773.0,409.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,27.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0
1500000US010179548003,281.0,85.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
1500000US010150011031,539.0,321.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1500000US010150024003,970.0,421.0,93.0,0.0,0.0,0.0,0.0,0.0,0.0,33.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
test1_2021 = pd.read_parquet(f'{DIR_2021}/acs_2021_X14_SCHOOL_ENROLLMENT_bg.parquet')

In [6]:
test1_2021.head()

,B14002_001E,B14002_002E,B14002_003E,B14002_004E,B14002_005E,B14002_006E,B14002_007E,B14002_008E,B14002_009E,B14002_010E,...,B14007I_010E,B14007I_011E,B14007I_012E,B14007I_013E,B14007I_014E,B14007I_015E,B14007I_016E,B14007I_017E,B14007I_018E,B14007I_019E
GEOID,,,,,,,,,,,,,,,,,,,,,
010010201001,691.0,296.0,46.0,4.0,0.0,4.0,0.0,0.0,0.0,31.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0
010010201002,1038.0,558.0,145.0,0.0,0.0,0.0,0.0,0.0,0.0,87.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0
010010202001,782.0,324.0,77.0,7.0,0.0,7.0,0.0,0.0,0.0,29.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
010010202002,1146.0,703.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,28.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
010010203001,2667.0,1256.0,329.0,0.0,0.0,0.0,25.0,25.0,0.0,117.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,5.0


In [7]:
test2 = pd.read_parquet(f'{DIR_2022}/acs_2022_X03_HISPANIC_OR_LATINO_ORIGIN_bg.parquet')

In [8]:
test2.head()

,B03002_E001,B03002_E002,B03002_E003,B03002_E004,B03002_E005,B03002_E006,B03002_E007,B03002_E008,B03002_E009,B03002_E010,...,B03002_E015,B03002_E016,B03002_E017,B03002_E018,B03002_E019,B03002_E020,B03002_E021,B03003_E001,B03003_E002,B03003_E003
GEOIDFQ,,,,,,,,,,,,,,,,,,,,,
1500000US010179548002,1375.0,1340.0,149.0,1191.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1375.0,1340.0,35.0
1500000US010179548004,797.0,767.0,450.0,314.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,30.0,0.0,0.0,0.0,797.0,767.0,30.0
1500000US010179548003,281.0,276.0,138.0,138.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,5.0,0.0,0.0,0.0,281.0,276.0,5.0
1500000US010150011031,560.0,560.0,560.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,560.0,560.0,0.0
1500000US010150024003,1003.0,1003.0,871.0,45.0,0.0,0.0,0.0,0.0,87.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1003.0,1003.0,0.0


In [9]:
test2_2021 = pd.read_parquet(f'{DIR_2021}/acs_2021_X03_HISPANIC_OR_LATINO_ORIGIN_bg.parquet')

In [10]:
test2_2021.head()

,B03002_001E,B03002_002E,B03002_003E,B03002_004E,B03002_005E,B03002_006E,B03002_007E,B03002_008E,B03002_009E,B03002_010E,...,B03002_015E,B03002_016E,B03002_017E,B03002_018E,B03002_019E,B03002_020E,B03002_021E,B03003_001E,B03003_002E,B03003_003E
GEOID,,,,,,,,,,,,,,,,,,,,,
010010201001,693.0,674.0,587.0,16.0,0.0,0.0,0.0,0.0,71.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,693.0,674.0,19.0
010010201002,1098.0,1089.0,887.0,155.0,0.0,38.0,0.0,0.0,9.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1098.0,1089.0,9.0
010010202001,844.0,834.0,336.0,421.0,0.0,0.0,0.0,0.0,77.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,844.0,834.0,10.0
010010202002,1166.0,1166.0,439.0,667.0,0.0,0.0,0.0,8.0,52.0,27.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1166.0,1166.0,0.0
010010203001,2685.0,2672.0,2011.0,531.0,0.0,26.0,0.0,0.0,104.0,0.0,...,0.0,0.0,0.0,7.0,6.0,6.0,0.0,2685.0,2672.0,13.0


It would be really cool if the 'E' moving was the only naming convention change with the new vintage

In [11]:
# New 2022-style ACS naming:
#   B02001_E001
#   B02001_M001
NEW_STYLE_ACS_RE = re.compile(r"^([A-Z0-9]+)_([EM])(\d{3})$", re.IGNORECASE)

# Canonical ACS naming:
#   B02001_001E
#   B02001_001M
CANONICAL_ACS_RE = re.compile(r"^([A-Z0-9]+)_(\d{3})([EM])$", re.IGNORECASE)

# Flexible GEOID-like matcher
GEOID_RE = re.compile(r"^GEOID([A-Z_].*)?$", re.IGNORECASE)


def read_parquet_columns(path: Path) -> list[str]:
    """Read parquet schema only, not the data."""
    schema = pq.ParquetFile(path).schema_arrow
    return schema.names


def canonicalize_column(col: str) -> str:
    """
    Normalize ACS variable names to canonical form.

    Examples:
      B02001_E001 -> B02001_001E
      B02001_M001 -> B02001_001M
      B02001_001E -> B02001_001E
    """
    c = col.strip()

    m = NEW_STYLE_ACS_RE.match(c)
    if m:
        stem, suffix, digits = m.groups()
        return f"{stem.upper()}_{digits}{suffix.upper()}"

    m = CANONICAL_ACS_RE.match(c)
    if m:
        stem, digits, suffix = m.groups()
        return f"{stem.upper()}_{digits}{suffix.upper()}"

    return c


def classify_column(col: str) -> str:
    c = col.strip()

    if GEOID_RE.match(c) or c in {"GEOIDFQ", "GEOID_Data"}:
        return "geoid_like"

    if NEW_STYLE_ACS_RE.match(c):
        return "acs_new_style"

    if CANONICAL_ACS_RE.match(c):
        return "acs_canonical"

    return "other"


def layer_key(path: Path) -> str:
    """
    Convert a filename into a year-agnostic layer key.
    """
    name = path.name

    if re.fullmatch(r"acs_\d{4}_bg\.parquet", name):
        return "ALL_BG"

    if re.fullmatch(r"acs_demographic_profile_\d{4}_bg\.parquet", name):
        return "DEMOGRAPHIC_PROFILE"

    m = re.fullmatch(r"acs_\d{4}_(.+?)_bg\.parquet", name)
    if m:
        return m.group(1)

    return name

## Sanity check: do the helper functions work?

Expected: 
- B02001_E001 should become B02001_001E
- B02001_M001 should become B02001_001M
- canonical names should stay unchanged
- GEOID-like columns should stay unchanged

In [12]:
test_cols = [
    "B02001_E001",
    "B02001_M001",
    "B02001_001E",
    "B19013_001E",
    "GEOID",
    "GEOIDFQ",
    "GEOID_Data",
    "NAME",
]

test_df = pd.DataFrame({
    "original": test_cols,
    "classification": [classify_column(c) for c in test_cols],
    "canonical": [canonicalize_column(c) for c in test_cols],
})

display(test_df)

,original,classification,canonical
0,B02001_E001,acs_new_style,B02001_001E
1,B02001_M001,acs_new_style,B02001_001M
2,B02001_001E,acs_canonical,B02001_001E
3,B19013_001E,acs_canonical,B19013_001E
4,GEOID,geoid_like,GEOID
5,GEOIDFQ,geoid_like,GEOIDFQ
6,GEOID_Data,geoid_like,GEOID_Data
7,NAME,other,NAME


## Compare the files between vintages

Just verifying comparable tables and identifying what is new

In [13]:
def parse_parquet_filename(path: Path) -> dict:
    """
    Parse known ACS parquet filenames into structured parts and normalize
    year-specific pieces so 2021 and 2022 comparable files align.
    """
    name = path.name

    # acs_demographic_profile_2022_bg.parquet
    m = re.match(
        r"^acs_demographic_profile_(\d{4})_(\w+)\.parquet$",
        name,
        flags=re.IGNORECASE,
    )
    if m:
        year, geography = m.groups()
        return {
            "file": name,
            "year": int(year),
            "kind": "demographic_profile",
            "x_code": None,
            "table_name": "demographic_profile",
            "geography": geography,
            "group_key": f"demographic_profile::{geography}",
            "sort_key": (9998, "demographic_profile", geography),
        }

    # acs_2022_X29_VOTING_AGE_POPULATION_bg.parquet
    m = re.match(
        r"^acs_(\d{4})_(X\d{2})_(.+)_(\w+)\.parquet$",
        name,
        flags=re.IGNORECASE,
    )
    if m:
        year, x_code, table_name, geography = m.groups()
        x_code = x_code.upper()
        return {
            "file": name,
            "year": int(year),
            "kind": "x_table",
            "x_code": x_code,
            "table_name": table_name,
            "geography": geography,
            "group_key": f"{x_code}::{table_name}::{geography}",
            "sort_key": (int(x_code[1:]), table_name, geography),
        }

    # acs_2022_ACS_2022_5YR_BG_bg.parquet
    # normalize ACS_2021_5YR_BG and ACS_2022_5YR_BG to ACS_5YR_BG
    m = re.match(
        r"^acs_(\d{4})_(ACS_\d{4}_5YR_[A-Z]+)_(\w+)\.parquet$",
        name,
        flags=re.IGNORECASE,
    )
    if m:
        year, source_name, geography = m.groups()
        source_name_norm = re.sub(r"ACS_\d{4}_5YR_", "ACS_5YR_", source_name, flags=re.IGNORECASE)
        return {
            "file": name,
            "year": int(year),
            "kind": "whole_gdb",
            "x_code": None,
            "table_name": source_name_norm,
            "geography": geography,
            "group_key": f"whole_gdb::{source_name_norm}::{geography}",
            "sort_key": (9996, source_name_norm, geography),
        }

    # acs_2022_bg.parquet
    m = re.match(
        r"^acs_(\d{4})_(\w+)\.parquet$",
        name,
        flags=re.IGNORECASE,
    )
    if m:
        year, geography = m.groups()
        return {
            "file": name,
            "year": int(year),
            "kind": "combined",
            "x_code": None,
            "table_name": "combined",
            "geography": geography,
            "group_key": f"combined::{geography}",
            "sort_key": (9997, "combined", geography),
        }

    return {
        "file": name,
        "year": None,
        "kind": "unknown",
        "x_code": None,
        "table_name": name,
        "geography": None,
        "group_key": f"unknown::{name}",
        "sort_key": (9999, name, ""),
    }

In [14]:
files_2021 = sorted(DIR_2021.glob("*.parquet"))
files_2022 = sorted(DIR_2022.glob("*.parquet"))

parsed_2021 = pd.DataFrame([parse_parquet_filename(p) for p in files_2021])
parsed_2022 = pd.DataFrame([parse_parquet_filename(p) for p in files_2022])

print(f"2021 parquet files: {len(parsed_2021)}")
print(f"2022 parquet files: {len(parsed_2022)}")

2021 parquet files: 26
2022 parquet files: 34


In [15]:
compare_files_df = (
    parsed_2021.rename(columns={"file": "file_2021", "year": "year_2021"})
    .merge(
        parsed_2022.rename(columns={"file": "file_2022", "year": "year_2022"}),
        on=["group_key", "kind", "x_code", "table_name", "geography", "sort_key"],
        how="outer",
    )
    .sort_values(["sort_key", "kind", "table_name", "group_key"])
    .reset_index(drop=True)
)

compare_files_df["exists_2021"] = compare_files_df["file_2021"].notna()
compare_files_df["exists_2022"] = compare_files_df["file_2022"].notna()

display(
    compare_files_df[
        [
            "file_2021",
            "file_2022",
            "exists_2021",
            "exists_2022",
        ]
    ]
)

,file_2021,file_2022,exists_2021,exists_2022
0,acs_2021_X01_AGE_AND_SEX_bg.parquet,acs_2022_X01_AGE_AND_SEX_bg.parquet,True,True
1,acs_2021_X02_RACE_bg.parquet,acs_2022_X02_RACE_bg.parquet,True,True
2,acs_2021_X03_HISPANIC_OR_LATINO_ORIGIN_bg.parquet,acs_2022_X03_HISPANIC_OR_LATINO_ORIGIN_bg.parquet,True,True
3,NaN,acs_2022_X04_ANCESTRY_bg.parquet,False,True
4,NaN,acs_2022_X05_FOREIGN_BORN_CITIZENSHIP_bg.parquet,False,True
5,NaN,acs_2022_X06_PLACE_OF_BIRTH_bg.parquet,False,True
6,acs_2021_X07_MIGRATION_bg.parquet,acs_2022_X07_MIGRATION_bg.parquet,True,True
7,acs_2021_X08_COMMUTING_bg.parquet,acs_2022_X08_COMMUTING_bg.parquet,True,True
8,acs_2021_X09_CHILDREN_HOUSEHOLD_RELATIONSHIP_b...,acs_2022_X09_CHILDREN_HOUSEHOLD_RELATIONSHIP_b...,True,True
9,NaN,acs_2022_X10_GRANDPARENTS_GRANDCHILDREN_bg.par...,False,True


This cell 

We want to see no columns changed for 2021, but many for 2022

# Housing characteristics??

Why is this table so much different? Are these really 100 new variables?

In [16]:
BUILD_ROOT = Path("../build")

file_2021 = BUILD_ROOT / "2021_bg" / "acs_2021_X25_HOUSING_CHARACTERISTICS_bg.parquet"
file_2022 = BUILD_ROOT / "2022_bg" / "acs_2022_X25_HOUSING_CHARACTERISTICS_bg.parquet"

print("2021 exists:", file_2021.exists(), file_2021)
print("2022 exists:", file_2022.exists(), file_2022)

2021 exists: True ../build/2021_bg/acs_2021_X25_HOUSING_CHARACTERISTICS_bg.parquet
2022 exists: True ../build/2022_bg/acs_2022_X25_HOUSING_CHARACTERISTICS_bg.parquet


In [17]:
def read_parquet_columns(path: Path) -> list[str]:
    return pq.ParquetFile(path).schema_arrow.names

cols_2021 = read_parquet_columns(file_2021)
cols_2022 = read_parquet_columns(file_2022)

print("n_cols_2021_raw:", len(cols_2021))
print("n_cols_2022_raw:", len(cols_2022))

n_cols_2021_raw: 871
n_cols_2022_raw: 971


Canonicalize the columns

In [18]:
df21 = pd.DataFrame({"raw_2021": cols_2021})
df21["canonical"] = df21["raw_2021"].map(canonicalize_column)

df22 = pd.DataFrame({"raw_2022": cols_2022})
df22["canonical"] = df22["raw_2022"].map(canonicalize_column)

In [19]:
aligned = (
    df21.merge(df22, on="canonical", how="outer")
    .sort_values("canonical")
    .reset_index(drop=True)
)

aligned["present_2021"] = aligned["raw_2021"].notna()
aligned["present_2022"] = aligned["raw_2022"].notna()

display(aligned.head(5))

,raw_2021,canonical,raw_2022,present_2021,present_2022
0,B25001_001E,B25001_001E,B25001_E001,True,True
1,B25002_001E,B25002_001E,B25002_E001,True,True
2,B25002_002E,B25002_002E,B25002_E002,True,True
3,B25002_003E,B25002_003E,B25002_E003,True,True
4,B25003A_001E,B25003A_001E,B25003A_E001,True,True


# Test the fix

Pushed a change on 4/14/26

## Synthetic Tests

In [20]:
pwd

'/home/dylan/projects/geosnap/build'

In [21]:
!pip install -e /home/dylan/projects/geosnap

Obtaining file:///home/dylan/projects/geosnap
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for geosnap (pyproject.toml) ... done
  Created wheel for geosnap: filename=geosnap-0.16.1.dev34+g2f9cb9274.d20260414-0.editable-py3-none-any.whl size=8639 sha256=d324b655c7466dd269e80cd78beb59a593c7e69b5f00ccc85ec83ddc2dd95bd5
  Stored in directory: /tmp/pip-ephem-wheel-cache-aoabk68n/wheels/50/05/f1/5afabb92124d2b3b9c0f2213aed7af8d580c81096a101a27a2
Successfully built geosnap
  Attempting uninstall: geosnap
    Found existing installation: geosnap 0.16.1.dev34+g2f9cb9274.d20260414
    Uninstalling geosnap-0.16.1.dev34+g2f9cb9274.d20260414:
      Successfully uninstalled geosnap-0.16.1.dev34+g2f9cb9274.d20260414


In [22]:
from geosnap.io.util import normalize_acs_vars

/home/dylan/mambaforge/envs/diss-data/lib/python3.13/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [23]:
tests = [
    "B02001e1",
    "B02001e12",
    "B02001_E001",
    "B02001_M001",
    "B02001_001E",
    "B02001_001M",
    "GEOID",
    "GEOIDFQ",
    "GEOID_Data",
    "geometry",
    "NAME",
]

for t in tests:
    print(f"{t:15} -> {normalize_acs_vars(t)}")

B02001e1        -> B02001_001E
B02001e12       -> B02001_012E
B02001_E001     -> B02001_001E
B02001_M001     -> B02001_001M
B02001_001E     -> B02001_001E
B02001_001M     -> B02001_001M
GEOID           -> GEOID
GEOIDFQ         -> GEOIDFQ
GEOID_Data      -> GEOID_Data
geometry        -> geometry
NAME            -> NAME


In [24]:
from geosnap.io.util import find_geoid_column

In [25]:
cases = [
    ["GEOID", "B01001_E001"],
    ["GEOIDFQ", "B01001_E001"],
    ["GEOID_Data", "B01001_E001"],
    ["GEOID20", "B01001_E001"],
    ["NAME", "B01001_E001"],
]

for cols in cases:
    print(cols, "->", find_geoid_column(cols))

['GEOID', 'B01001_E001'] -> GEOID
['GEOIDFQ', 'B01001_E001'] -> GEOIDFQ
['GEOID_Data', 'B01001_E001'] -> GEOID_Data
['GEOID20', 'B01001_E001'] -> GEOID20
['NAME', 'B01001_E001'] -> None


/home/dylan/projects/geosnap/geosnap/io/util.py:144: UserWarning: No GEOID-like column found. Columns are: ['NAME', 'B01001_E001']
  warn(f"No GEOID-like column found. Columns are: {list(columns)}")


In [26]:
cols = pd.Index([
    # old style
    "B02001e1", "B02001e12",

    # 2022 style
    "B02001_E001", "B02001_M001",

    # canonical style
    "B02001_001E", "B02001_001M",

    # noise / non-ACS
    "GEOID", "GEOIDFQ", "NAME", "geometry",
    "random_column", "B02001X001",
])

df = pd.DataFrame(columns=cols)

print("All columns:")
print(list(df.columns))

All columns:
['B02001e1', 'B02001e12', 'B02001_E001', 'B02001_M001', 'B02001_001E', 'B02001_001M', 'GEOID', 'GEOIDFQ', 'NAME', 'geometry', 'random_column', 'B02001X001']


In [27]:
candidate_cols = df.columns[
    df.columns.str.match(r"^[A-Za-z0-9]+e\d+$", na=False)               # old style
    | df.columns.str.match(r"^[A-Za-z0-9]+_[EM]\d{3}$", na=False)       # 2022 style
    | df.columns.str.match(r"^[A-Za-z0-9]+_\d{3}[EM]$", na=False)       # canonical style
]

print("Selected columns:")
print(list(candidate_cols))
# should not contain any GEOID or 'geometry' or 'random_column'

Selected columns:
['B02001e1', 'B02001e12', 'B02001_E001', 'B02001_M001', 'B02001_001E', 'B02001_001M']


# Examine output from `process_acs`
After turning the new conversion function loose, I applied the `process_acs` function on the resulting combined demographic profile. Let's look at both here

In [34]:
demographic_profile = pd.read_parquet('2022_bg/acs_demographic_profile_2022_bg.parquet')
processed_acs = pd.read_parquet('2022_bg/acs_2022_bg_processed.parquet')

In [37]:
display(demographic_profile)

,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,...,B01002H_003E,B01002H_003M,B01002I_001E,B01002I_001M,B01002I_002E,B01002I_002M,B01002I_003E,B01002I_003M,B01003_001E,B01003_001M
010179548002,01,017,954800,2,Block Group 2,G5030,S,1094218.0,0.0,+32.8662046,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
010179548004,01,017,954800,4,Block Group 4,G5030,S,2392140.0,0.0,+32.8482537,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
010179548003,01,017,954800,3,Block Group 3,G5030,S,902949.0,0.0,+32.8577594,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
010150011031,01,015,001103,1,Block Group 1,G5030,S,2346322.0,94061.0,+33.5892886,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
010150024003,01,015,002400,3,Block Group 3,G5030,S,38223047.0,173264.0,+33.9079142,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1500000US720210302002,None,None,None,None,None,None,None,NaN,NaN,None,...,-666666666.0,-222222222.0,57.6,13.3,53.7,42.9,58.5,17.3,597.0,239.0
1500000US720210314012,None,None,None,None,None,None,None,NaN,NaN,None,...,-666666666.0,-222222222.0,58.4,12.9,56.5,13.9,63.3,15.0,977.0,285.0
1500000US720210312021,None,None,None,None,None,None,None,NaN,NaN,None,...,-666666666.0,-222222222.0,44.5,6.9,49.2,14.6,43.7,5.3,1837.0,372.0
1500000US720531504003,None,None,None,None,None,None,None,NaN,NaN,None,...,-666666666.0,-222222222.0,38.7,13.0,33.8,11.7,47.1,15.1,1115.0,365.0


In [38]:
display(processed_acs)

,n_total_housing_units,n_vacant_housing_units,n_occupied_housing_units,n_owner_occupied_housing_units,n_renter_occupied_housing_units,n_housing_units_multiunit_structures_denom,n_total_housing_units_sample,median_home_value,median_contract_rent,n_occupied_housing_units_sample,...,p_owner_occupied_units,p_married,p_female_headed_families,p_nonhisp_white_persons,p_nonhisp_black_persons,p_hispanic_persons,p_native_persons,p_hawaiian_persons,p_veterans,geometry
GEOID,,,,,,,,,,,,,,,,,,,,,
010179548002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
010179548004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
010179548003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
010150011031,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
010150024003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1500000US720210302002,332.0,104.0,228.0,159.0,69.0,332.0,332.0,155500.0,582.0,228.0,...,68.674699,9.854015,15.000000,0.000000,0.0,100.000000,0.0,0.0,9.882747,None
1500000US720210314012,599.0,155.0,444.0,336.0,108.0,599.0,599.0,132400.0,519.0,444.0,...,74.123539,19.549642,0.000000,0.000000,0.0,100.000000,0.0,0.0,5.424770,None
1500000US720210312021,821.0,129.0,692.0,554.0,138.0,821.0,821.0,103000.0,460.0,692.0,...,84.287454,10.917816,9.859155,0.000000,0.0,98.530212,0.0,0.0,3.647251,None
